# 02 — Target Domain: Zero-Shot Transfer & Adaptation

Two things happen here:
1. Evaluate the **untouched, source-trained** model directly on the target
   domain — this is "the drop."
2. Run **Method A** (full fine-tune) and **Method B** (adapter + MMD) on the
   small target training split and re-evaluate.

In [ ]:
import sys
sys.path.append("..")

import torch, yaml
from torch.utils.data import DataLoader

from src.datasets import build_target_dataset, build_source_dataset
from src.models import VisionLanguageRetriever
from src.evaluation import evaluate
from src.adaptation import run_finetune, run_adapter_mmd, build_source_embedding_bank

device = "cuda" if torch.cuda.is_available() else "cpu"

with open("../configs/model.yaml") as f:
    model_cfg = yaml.safe_load(f)
with open("../configs/data.yaml") as f:
    data_cfg = yaml.safe_load(f)

## Step 1 — Zero-shot transfer (the performance drop)

In [ ]:
model = VisionLanguageRetriever.from_config(model_cfg)
model.load_state_dict(torch.load("../experiments/baseline/checkpoints/best.pt", map_location=device))
model.to(device).eval()
tokenizer = model.tokenizer

dl_cfg = data_cfg["dataloader"]
target_test_ds = build_target_dataset(data_cfg, split="test", tokenizer=tokenizer)
target_test_loader = DataLoader(target_test_ds, batch_size=dl_cfg["batch_size"], shuffle=False)

zero_shot_result = evaluate(model, target_test_loader, device=device,
                             out_file="../results/tables/target_zero_shot_eval.json")
zero_shot_result["metrics"]

Compare `recall@1` here against `01_source_domain.ipynb`'s result — that difference **is** the performance drop this project studies.

## Step 2 — Method A: full fine-tune

In [ ]:
with open("../configs/adapt_finetune.yaml") as f:
    ft_cfg = yaml.safe_load(f)

model_a = VisionLanguageRetriever.from_config(model_cfg)
model_a.load_state_dict(torch.load("../experiments/baseline/checkpoints/best.pt", map_location=device))

target_train_ds = build_target_dataset(data_cfg, split="train", tokenizer=tokenizer, few_shot=True)
target_train_loader = DataLoader(target_train_ds, batch_size=dl_cfg["batch_size"], shuffle=True)
print(f"Method A trains on only {len(target_train_ds)} target pairs (few-shot, by design)")

model_a, history_a = run_finetune(model_a, target_train_loader, ft_cfg["train"], device=device)
result_a = evaluate(model_a, target_test_loader, device=device, out_file="../results/tables/method_a_eval.json")
result_a["metrics"]

## Step 3 — Method B: adapter + MMD alignment

In [ ]:
with open("../configs/adapt_adapter_mmd.yaml") as f:
    mmd_cfg = yaml.safe_load(f)

model_b = VisionLanguageRetriever.from_config(model_cfg, adapter_override=mmd_cfg["adapter_override"])
model_b.load_state_dict(torch.load("../experiments/baseline/checkpoints/best.pt", map_location=device), strict=False)

source_ds = build_source_dataset(data_cfg, split="train", tokenizer=tokenizer)
source_loader = DataLoader(source_ds, batch_size=dl_cfg["batch_size"], shuffle=True)
source_bank = build_source_embedding_bank(model_b, source_loader, mmd_cfg["train"], device=device)

model_b, history_b = run_adapter_mmd(model_b, target_train_loader, source_bank, mmd_cfg["train"], device=device)
result_b = evaluate(model_b, target_test_loader, device=device, out_file="../results/tables/method_b_eval.json")
result_b["metrics"]

## Step 4 — Side-by-side

In [ ]:
import pandas as pd
pd.DataFrame([
    {"run": "Zero-shot (drop)", **zero_shot_result["metrics"]},
    {"run": "Method A (fine-tune)", **result_a["metrics"]},
    {"run": "Method B (adapter+MMD)", **result_b["metrics"]},
])[["run", "recall@1", "recall@5", "recall@10", "mAP"]]

Continue to `03_error_analysis.ipynb` to see *which* queries recovered and why.